# 3D-SynTree: Execution & Training Engine
**Structure-Based Molecular Design via Reaction-Constrained Synthon Assembly**

This notebook acts purely as an execution and download wrapper for the `3d-syntree` repository.
All business logic, dataset streaming, and modeling reside strictly in the codebase.

**Workflow:**
1. Environment & HF Authentication
2. Clone Repository
3. **Dedicated Update Cell (Always sync to latest Git commit)**
4. Dependency Installation
5. Download Pre-filtered Dataset & Synthon Assets (from HF Dataset Repo)
6. Hardware Verification
7. Execute Multi-Day Resilient Training (Checkpoints sync to HF Model Repo)
8. Verification & Diagnostics
9. Stage 2 Chemistry-Constrained PPO Fine-Tuning
10. Comparative Benchmark Battery

In [ ]:
# CELL 1: Environment & Hugging Face Authentication
import os, json

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        try:
            from kaggle_secrets import UserSecretsClient
            HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            HF_TOKEN = ""

if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("\nEnter your HuggingFace WRITE token (blank = local checkpoints only): ").strip()

os.environ["HF_TOKEN"] = HF_TOKEN

if HF_TOKEN:
    from huggingface_hub import whoami
    try:
        info = whoami(token=HF_TOKEN)
        user = info.get("name", "?")
        role = (info.get("auth", {}).get("accessToken", {}) or {}).get("role", "unknown")
        print(f"HF token verified for '{user}' (reported role: '{role}').")
    except Exception as e:
        raise RuntimeError(f"HF token validation failed: {e}")
else:
    print("HF_TOKEN is empty. Dataset reads remain public; model checkpoints will stay local.")

RUNTIME_CONFIG = {
    "repo_url": "https://github.com/Vtheonly/3d-syntree.git",
    "branch": "main",
    "hf_dataset_repo_id": "JJKK1212/3d-syntree-multidataset",
    "hf_model_repo_id": "JJKK1212/3d-syntree-checkpoints",
    "config_override": {
        "huggingface": {
            "enabled": bool(HF_TOKEN),
            "repo_id": "JJKK1212/3d-syntree-checkpoints",
            "push_every_n_epochs": 2,
            "private": False
        },
        "data": {
            "backend": "huggingface",
            "use_hf_dataset": True,
            "hf_dataset_repo": "JJKK1212/3d-syntree-multidataset",
            "data_dir": "./data/crossdocked",
            "synthon_catalog_path": "./data/enamine_3d_subset.parquet",
            "huggingface": {
                "repo_id": "JJKK1212/3d-syntree-multidataset",
                "revision": "main",
                "cache_dir": "./hf_cache",
                "max_cached_shards": 2
            }
        },
        "training": {
            "time_budget_hours": 11.5,
            "auto_scale": {
                "enabled": True,
                "target_vram_fraction": 0.85,
                "max_batch_size": 2048,
                "scale_model": True
            }
        }
    }
}

with open("runtime_config.json", "w") as f:
    json.dump(RUNTIME_CONFIG, f, indent=2)
print("Runtime configuration profile generated successfully.")

In [ ]:
# CELL 2: Clone Repository (First-Time Setup)
import os

REPO_DIR = "3d-syntree"
REPO_URL = RUNTIME_CONFIG["repo_url"]
BRANCH = RUNTIME_CONFIG["branch"]

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} (branch: {BRANCH})...")
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} directory already exists.")

%cd {REPO_DIR}

In [ ]:
# CELL 3: Dedicated Force-Update Cell
import os

BRANCH = RUNTIME_CONFIG["branch"]

print(f"Fetching latest commits and hard-resetting to origin/{BRANCH}...")
!git fetch --all --prune
!git checkout {BRANCH}
!git reset --hard origin/{BRANCH}
!git clean -fd

print("\nCurrent commit:")
!git log -1 --oneline

In [ ]:
# CELL 4: Install Dependencies & Setup GNINA Docking Oracle
!wget -q https://github.com/gnina/gnina/releases/download/v1.1/gnina -O /usr/local/bin/gnina && chmod +x /usr/local/bin/gnina
!/usr/local/bin/gnina --version || echo 'gnina unavailable'

print("Installing requirements and package in editable mode...")
!pip install --quiet --upgrade pip
!pip install --quiet -r requirements.txt
!pip install --quiet -e .

import rdkit, torch, torch_geometric, huggingface_hub
print(
    f"Environment Verified: PyTorch {torch.__version__} | "
    f"PyG {torch_geometric.__version__} | RDKit {rdkit.__version__} | "
    f"Hugging Face Hub {huggingface_hub.__version__}"
)

In [ ]:
# CELL 5: Sync Clean Sharded Dataset & Synthon Catalog from HF Dataset Hub
DATASET_REPO = RUNTIME_CONFIG["hf_dataset_repo_id"]
print(f"Synchronizing preprocessed assets from HF Dataset: {DATASET_REPO}...")

!python scripts/download_assets.py \
    --dataset-repo {DATASET_REPO} \
    --dataset-revision main \
    --output-dir ./data

print("Dataset manifests, exact synthon catalog, and asset provenance verified.")

In [ ]:
# CELL 6: Hardware Verification
import json, torch
from syntree.utils.hardware import configure_runtime_environment, free_vram_bytes

device_info = configure_runtime_environment()
print("Hardware Execution Profile:")
print(json.dumps(device_info, indent=2))

assert device_info["device"].startswith("cuda"), (
    "ERROR: No GPU detected! Go to Runtime > Change runtime type > GPU."
)

free_gb = free_vram_bytes() / 1024**3
print(f"Free VRAM: {free_gb:.2f} GB -> Target 85% utilization: {0.85 * free_gb:.2f} GB")

In [ ]:
# CELL 6.5: Training Mode Configuration (Fresh Run vs Auto-Resume)
# Set FRESH_TRAINING = True if you want to start a brand-new run from epoch 0 (clears old checkpoints).
# Set FRESH_TRAINING = False to auto-resume from the latest checkpoint (disconnection immunity).
FRESH_TRAINING = False

TRAIN_FLAG = "--fresh" if FRESH_TRAINING else "--resume-auto"
print(f"Training mode: {'FRESH RUN (--fresh, starts from epoch 0)' if FRESH_TRAINING else 'AUTO-RESUME (--resume-auto)'}")

In [ ]:
# CELL 7: Launch Resilient Training
print(f"Starting 3D-SynTree training engine ({TRAIN_FLAG})...")

!python main.py \
    --mode train \
    --config configs/train_colab_12h.json \
    --runtime-config ../runtime_config.json \
    $TRAIN_FLAG

In [ ]:
# CELL 8: Status & Checkpoint Sync Verification
import json, os
from syntree.utils.checkpoint import verify_hf_sync

MODEL_REPO = RUNTIME_CONFIG["hf_model_repo_id"]
status = verify_hf_sync(MODEL_REPO)

print("=== SESSION STATUS ===")
print(f"Hugging Face Model Repo: {MODEL_REPO}")
print(f"Latest Remote Checkpoint: {status['latest_remote_checkpoint']}")
print(f"Total Epochs Completed:   {status['epochs_completed']}")
print(f"Sync Operational:         {status['sync_ok']}")
if "error" in status and status["error"]:
    print(f"Sync Alert: {status['error']}")

progress_path = "experiments/checkpoints/progress.json"
if os.path.exists(progress_path):
    with open(progress_path) as f:
        progress = json.load(f)
    print(f"Local sequential progress: {progress.get('completed_epochs', [])}")

if os.path.exists("experiments/latest_metrics.json"):
    with open("experiments/latest_metrics.json") as f:
        print("\nLatest Validation Metrics:")
        print(json.dumps(json.load(f), indent=2))

## Stage 2: Chemistry-Constrained PPO Fine-Tuning
Optimizes the policy using the GNINA/Vina docking oracle and soft Lennard-Jones contact energy without relaxing the Enamine reaction grammar.

In [ ]:
# CELL 9: Stage 2 PPO
print("Starting chemistry-constrained PPO fine-tuning...")
!python main.py \
    --mode rl \
    --config configs/rl_colab_12h.json \
    --pocket-dir ./data/test_pockets \
    --resume-auto

## Comparative Evaluation Battery
Executes PoseBusters physical validity, Fsp3, uniqueness, and docking benchmarks across target pockets.

In [ ]:
# CELL 10: Comparative Benchmark
!python scripts/run_comparative_benchmark.py \
    --manifest ./benchmarks/targets.jsonl \
    --outputs ./benchmarks/outputs \
    --methods 3d-syntree \
    --limit 100